# Sum of Squares — Jane Street, January 2014

> Place a digit in each of the 25 spots in the 5×5 grid, so that each 5-digit number
> (leading zeroes are ok) reading across and reading down is divisible by the number
> outside the grid, **maximizing the sum of the 25 digits** you enter.

|       | ÷6 | ÷7 | ÷8 | ÷9 | ÷10 |
|-------|----|----|----|----|-----|
| **÷1**  | ·  | ·  | ·  | ·  | ·   |
| **÷2**  | ·  | ·  | ·  | ·  | ·   |
| **÷3**  | ·  | ·  | ·  | ·  | ·   |
| **÷4**  | ·  | ·  | ·  | ·  | ·   |
| **÷5**  | ·  | ·  | ·  | ·  | ·   |

Answer: `(sum all 25 digits)`.

---

> **A note on style.** Every loop here is written out explicitly rather than compressed into a
> list comprehension, and every test spells out its comparison (`if value % divisor != 0:` rather
> than `if value % divisor:`). It is more lines than an experienced Python programmer would write.
> That is deliberate: in solver work, code you can read at a glance is code you can trust and
> debug, and the compressed forms buy nothing here.

## Goals

I wanted to learn how to use solvers, so going to old Jane Street puzzles was a good way to do
this.  The goal here isn't to solve the problem, which is trivial with modern solvers, but rather
to learn how solvers work, how to specify them, how to add human derived constraints, etc...

## AI Disclaimer

I asked AI to essentially make this worksheet a tutorial for using googles CP-SAT.  Its use case
is to simply work through the notebook, and I plan to annotate all the code with my understanding
of what its doing.  None of the code is my own, and the majority of the writing is AI explanation.

## 1. How a constraint solver actually works

Skip this if you want to just run code, but it's ten minutes that will save you hours later.

### The naive picture is hopeless

There are $10^{25}$ ways to fill this grid. At a billion grids per second, checking them all
takes about 300 million years. No amount of hardware fixes that. So a solver cannot be
"a fast loop" — it has to be doing something structurally different.

### The three ingredients

A **constraint programming (CP)** model has exactly three parts:

- **Decision variables**, each with a *domain* — a finite set of values it's allowed to take.
  Here: 25 variables, each with domain $\{0,1,\dots,9\}$.
- **Constraints**, which are relations that must hold between variables.
  Here: 10 divisibility relations.
- Optionally an **objective** to maximize or minimize. Here: the sum of the 25 digits.

You describe *what* must be true. You never write the search. That's the whole pitch.

### The search loop

Under the hood the solver runs a loop that looks like this:

```
while some variable still has more than one value in its domain:
    1. BRANCH    pick a variable, split its domain (e.g. "is cell[0][0] >= 5?")
    2. PROPAGATE for every constraint touching a changed variable, delete values
                 from its neighbours' domains that can no longer participate in
                 any solution. Repeat until nothing changes.
    3. if some domain became empty:
           CONFLICT -> back up, and LEARN a reason so this branch is never retried
```

**Propagation is where the leverage is.** A constraint isn't a test you run at the end — it's an
active agent that deletes values the moment it can prove they're dead. Fix the bottom-left cell
to 3, and the "column 1 is divisible by 6" constraint instantly deletes values from the other
four cells of column 1, which shrinks what column-mates can be, which cascades into the row
constraints, and so on. One assignment can wipe out $10^{20}$ grids without ever looking at them.

**Conflict learning is the other half.** When the solver hits a contradiction it doesn't just
back up one step — it analyses *which* assignments caused it and records a new constraint (a
"learned clause") forbidding that combination anywhere else in the tree. This is the CDCL
algorithm from SAT solving, and it's why modern solvers beat hand-written backtracking by orders
of magnitude: they get monotonically smarter about the same problem as they search it.

### Where CP-SAT sits

We'll use Google OR-Tools' **CP-SAT**. The name is the architecture: it takes your integer
model, compiles much of it into a Boolean SAT problem, and runs a CDCL SAT core *alongside*
specialised integer propagators and an LP relaxation, sharing information between them. In
practice it runs several different search strategies in parallel and lets them trade discoveries.

It's the right default for this class of puzzle: finite integer domains, an objective, and native
support for enumerating every solution. It also has the best documentation and the largest
community of any Python CP tool, which matters a lot when you're learning.

### One vocabulary note

An **optimization** solver gives you two numbers, not one: the best solution it has *found*, and
the best value it can still *prove* might exist (the bound). When those two meet, the search is
over and the answer is provably optimal. Watching that gap close is how you read a solver log.

## 2. Setup

Environment (already done for this repo — here for reference / other machines):

```bash
uv venv                            # .venv/ at the repo root
uv pip install ortools ipykernel   # VS Code auto-detects .venv as a kernel
```

Pick the `.venv` kernel in the top-right of this notebook, then Run All.

In [38]:
import math

from ortools.sat.python import cp_model

print("OR-Tools", __import__("ortools").__version__)

OR-Tools 9.15.6755


In [39]:
# --- the puzzle's clues -------------------------------------------------
ROW_DIVISORS = [1, 2, 3, 4, 5]  # top row to bottom row
COL_DIVISORS = [6, 7, 8, 9, 10]  # left column to right column
N = 5

# Place values: the digit in position i of a 5-digit number is worth PLACE[i].
PLACE = [10_000, 1_000, 100, 10, 1]

### Three small helpers on plain grids

These work on ordinary lists of integers, not on solver variables. They're used by the checker
below and by the printing function, so the "what is the value of row 3?" logic lives in exactly
one place.

In [40]:
def value_of_row(grid, r):
    """The 5-digit number formed by reading row r left to right."""
    total = 0
    for c in range(N):
        total = total + grid[r][c] * PLACE[c]
    return total


def value_of_column(grid, c):
    """The 5-digit number formed by reading column c top to bottom."""
    total = 0
    for r in range(N):
        total = total + grid[r][c] * PLACE[r]
    return total


def digit_sum(grid):
    """The sum of all 25 digits — the quantity the puzzle asks us to maximize."""
    total = 0
    for r in range(N):
        for c in range(N):
            total = total + grid[r][c]
    return total

In [41]:
def show(grid, title=""):
    """Print a grid alongside its divisor clues and the value of every row and column."""
    if title != "":
        print(title)

    # Header line: the column divisors.
    header = "        "
    for divisor in COL_DIVISORS:
        header = header + f"÷{divisor:<2}  "
    print(header)

    # One line per row: the row's divisor, its digits, and the number they form.
    for r in range(N):
        digit_text = ""
        for c in range(N):
            digit_text = digit_text + f"{grid[r][c]}   "
        print(f"  ÷{ROW_DIVISORS[r]:<2}   {digit_text} = {value_of_row(grid, r):05d}")

    print()
    for c in range(N):
        print(f"  col {c + 1} (÷{COL_DIVISORS[c]:<2}) = {value_of_column(grid, c):05d}")

    print(f"\n  SUM OF DIGITS = {digit_sum(grid)}")

## 3. Write the checker before the model

**This is the single most important habit in solver work.** Write a plain-Python function that
takes a filled grid and says whether it's legal — no solver involved. Then test it on a grid you
already know is legal.

Why it matters: a constraint model is *write-only code*. If you mis-encode a rule, the solver
will happily return a confident, beautifully-formatted, completely wrong answer, and nothing will
look broken. The independent checker is the only thing standing between you and submitting
garbage. It also doubles as your final verification step in section 8.

Jane Street handed us a free test case: the example grid in the puzzle image, which they say sums
to 100.

Note that `check` returns a **list of problems** rather than `True`/`False`. A bare `False` tells
you nothing when you're debugging 25 cells; a list naming the broken rule and its value points
you straight at the mistake.

In [42]:
EXAMPLE = [
    [1, 6, 2, 3, 5],
    [5, 2, 4, 6, 0],
    [0, 4, 8, 9, 3],
    [2, 4, 8, 6, 8],
    [4, 7, 0, 3, 0],
]


def check(grid):
    """Return a list of rule violations. An empty list means the grid is legal."""
    problems = []

    for r in range(N):
        divisor = ROW_DIVISORS[r]
        value = value_of_row(grid, r)
        if value % divisor != 0:
            problems.append(f"row {r + 1} = {value:05d} is not divisible by {divisor}")

    for c in range(N):
        divisor = COL_DIVISORS[c]
        value = value_of_column(grid, c)
        if value % divisor != 0:
            problems.append(f"col {c + 1} = {value:05d} is not divisible by {divisor}")

    return problems


def print_check(grid):
    """Print whether a grid is legal, naming the broken rules if it isn't."""
    problems = check(grid)
    if len(problems) == 0:
        print("checker says: ✓ legal")
    else:
        print("checker says:", problems)

In [43]:
# If either of these fails, the grid was transcribed wrongly from the puzzle image.
# The second argument to `assert` is the message shown on failure — here, the list of
# broken rules, which is far more useful than a bare AssertionError.
assert check(EXAMPLE) == [], check(EXAMPLE)
assert digit_sum(EXAMPLE) == 100

print("✓ checker agrees with the puzzle's own example grid (sum 100)")
show(EXAMPLE, "\nThe example from the puzzle page:")

✓ checker agrees with the puzzle's own example grid (sum 100)

The example from the puzzle page:
        ÷6   ÷7   ÷8   ÷9   ÷10  
  ÷1    1   6   2   3   5    = 16235
  ÷2    5   2   4   6   0    = 52460
  ÷3    0   4   8   9   3    = 04893
  ÷4    2   4   8   6   8    = 24868
  ÷5    4   7   0   3   0    = 47030

  col 1 (÷6 ) = 15024
  col 2 (÷7 ) = 62447
  col 3 (÷8 ) = 24880
  col 4 (÷9 ) = 36963
  col 5 (÷10) = 50380

  SUM OF DIGITS = 100


## 4. The base model

Now the model. Three steps, mirroring section 1: variables, constraints, objective.

### Variables

25 integer variables with domain $\{0,\dots,9\}$. `new_int_var(lower, upper, label)` — the bounds
*are* the initial domain, so keep them as tight as you honestly can. Sloppy bounds are the most
common cause of a model that's mysteriously slow.

Two things share the word "cell" below, and it's worth separating them now:

- `cells` is an ordinary **Python list of lists** holding references to the variables. The 5×5
  *shape* exists only here — the model stores its variables as one flat numbered list.
- `"cell[2][3]"` is a **label string** attached to a variable for display in solver logs. It has
  no effect on solving and you cannot look a variable up by it.

You need to keep `cells` around because it is your only map from "row 2, column 3" to the actual
variable object — needed to build constraints and to read the answer back out afterwards.

### Turning digits into numbers

A row's value is a **linear expression** in its digit variables:

$$\text{row}_r = 10000\,d_{r,0} + 1000\,d_{r,1} + 100\,d_{r,2} + 10\,d_{r,3} + d_{r,4}$$

Ordinary Python arithmetic on variables *builds an expression object*; it does not compute a
number. Anywhere a constraint expects a value, you can hand it one of these.

### Encoding "divisible by k" — the important bit

The obvious encoding is `model.add_modulo_equality(0, row_value, divisor)`. It works.
**Don't use it.**

The better encoding introduces a **quotient variable** $q$ and asserts

$$\text{row}_r = k \cdot q, \qquad 0 \le q \le \lfloor 99999/k \rfloor$$

Why this is strictly better:

- It is a **pure linear equality**. CP-SAT solves a linear relaxation of your model in the
  background to compute bounds; linear constraints feed straight into it, and `modulo` does not.
- Bound propagation is much sharper. From $q \le 14285$ the solver immediately knows
  $\text{row} \le 99995$, and that flows back into the digit domains. The modulo propagator is
  generic and reasons far more weakly.
- $q$'s tight domain is itself free information you'd otherwise be throwing away.

**The general lesson, worth internalising:** when you have a choice of encodings, prefer linear
over nonlinear, and prefer nonlinear-but-specialised over generic. Two models with identical
solution sets can differ by orders of magnitude in solve time purely because of how well the
solver can *reason* about their form.

Note also that every quotient gets its **own** variable. Sharing one across two divisibility
rules would assert both `value == 3 * q` and `value == 5 * q`, forcing the value to zero.

### Objective

`model.maximize(...)` over the sum of all 25 digits. Note the trivial upper bound: 25 digits × 9
= 225. Keep that number in mind — it's the ceiling the solver's bound will descend from.

In [44]:
def new_grid_variables(model, label_prefix="cell"):
    """Create 25 digit variables (domain 0..9) and return them as a 5x5 list of lists.

    The list comprehension version of this is a one-liner, but written out you can see
    exactly where the list is created and where each variable is appended to it.
    """
    grid = []
    for r in range(N):
        row = []
        for c in range(N):
            variable = model.new_int_var(0, 9, f"{label_prefix}[{r}][{c}]")
            row.append(variable)
        grid.append(row)
    return grid

In [45]:
def row_expression(cells, r):
    """The 5-digit number formed by row r, as a linear expression over the variables."""
    expression = 0
    for c in range(N):
        expression = expression + cells[r][c] * PLACE[c]
    return expression


def column_expression(cells, c):
    """The 5-digit number formed by column c, as a linear expression over the variables."""
    expression = 0
    for r in range(N):
        expression = expression + cells[r][c] * PLACE[r]
    return expression


def row_digit_sum_expression(cells, r):
    """The sum of the five digits in row r, as a linear expression."""
    expression = 0
    for c in range(N):
        expression = expression + cells[r][c]
    return expression


def column_digit_sum_expression(cells, c):
    """The sum of the five digits in column c, as a linear expression."""
    expression = 0
    for r in range(N):
        expression = expression + cells[r][c]
    return expression


def total_expression(cells):
    """The sum of all 25 digit variables — the objective."""
    expression = 0
    for r in range(N):
        for c in range(N):
            expression = expression + cells[r][c]
    return expression

In [46]:
def force_divisible(model, expression, divisor, largest_value, label):
    """Require that `expression` is a multiple of `divisor`.

    Written as `expression == divisor * quotient` rather than as a modulo constraint,
    because a plain linear equation propagates far better (see the notes above).

    `largest_value` is the biggest the expression could possibly be; it sets the
    quotient's upper bound. CP-SAT has no unbounded integer variable, so some bound
    must be given — and a tight one is free information for the solver.
    """
    largest_quotient = largest_value // divisor
    quotient = model.new_int_var(0, largest_quotient, f"quotient_{label}")
    model.add(expression == divisor * quotient)
    return quotient


def build_base():
    """The puzzle's rules and nothing else. Returns (model, cells, total_expression)."""
    model = cp_model.CpModel()
    cells = new_grid_variables(model)

    for r in range(N):
        divisor = ROW_DIVISORS[r]
        force_divisible(model, row_expression(cells, r), divisor, 99_999, f"row{r}")

    for c in range(N):
        divisor = COL_DIVISORS[c]
        force_divisible(model, column_expression(cells, c), divisor, 99_999, f"col{c}")

    return model, cells, total_expression(cells)


print(
    "build_base() defined — 25 digit variables, 10 quotient variables, 10 constraints"
)

build_base() defined — 25 digit variables, 10 quotient variables, 10 constraints


In [62]:
# What the model actually contains. Worth reading once: 35 variables (25 digits plus
# 10 quotients) and 10 linear constraints, exactly as described above.
model, cells, total = build_base()
print(model.model_stats())

satisfaction model '': (model_fingerprint: 0x22506e665ae9bca5)
#Variables: 35 (26 primary variables)
  - 25 in [0,9]
  - 1 in [0,9999]
  - 1 in [0,11111]
  - 1 in [0,12499]
  - 1 in [0,14285]
  - 1 in [0,16666]
  - 1 in [0,19999]
  - 1 in [0,24999]
  - 1 in [0,33333]
  - 1 in [0,49999]
  - 1 in [0,99999]
#kLinearN: 10 (#terms: 60)


## 5. Solve it, and read the output like a professional

A solver run gives you far more than an answer. Learn to read all of it:

| Field | What it means |
|---|---|
| `OPTIMAL` | Solution found **and proved** best possible. The gold standard. |
| `FEASIBLE` | A valid solution, but the solver ran out of time before proving it optimal. |
| `INFEASIBLE` | **Proved** no solution exists. Usually means *you* made a modeling error. |
| `objective_value` | The best solution found. |
| `best_objective_bound` | The best value that could still conceivably exist. |
| `num_branches` | How many decisions the search made. Low = propagation did the work. |
| `num_conflicts` | How many dead ends it hit and learned from. |
| `wall_time` | Seconds. |

The pair `objective_value` and `best_objective_bound` is the one people ignore and shouldn't.
The solver is squeezing from both sides: raising the best-found value, lowering the provable
ceiling. **When they meet, the gap is zero and you have a proof.** A `FEASIBLE` result with a
bound of 205 and a value of 198 tells you something valuable — there might be a 205 out there,
keep looking. Always look at both.

Two habits: always set `max_time_in_seconds` so a bad model can't hang your kernel, and set
`num_workers` to use your cores (CP-SAT runs a *portfolio* of different search strategies in
parallel and shares learned clauses between them — it's not just data-parallel).

In [49]:
def read_grid_from_solver(solver, cells):
    """Pull the assigned digit out of every variable, as a plain 5x5 list of ints."""
    grid = []
    for r in range(N):
        row = []
        for c in range(N):
            row.append(solver.value(cells[r][c]))
        grid.append(row)
    return grid


def solve(model, cells, time_limit=60, workers=8, log=False, hint=None):
    """Solve the model. Returns (solver, status, grid) where grid is None if unsolved."""
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_workers = workers
    solver.parameters.log_search_progress = log

    # A hint is a suggested starting solution. Note this MUTATES the model rather than
    # the solver, so a hinted model stays hinted if you reuse it.
    if hint is not None:
        for r in range(N):
            for c in range(N):
                model.add_hint(cells[r][c], hint[r][c])

    status = solver.solve(model)

    # Only read variable values when a solution actually exists — reading them after an
    # INFEASIBLE or UNKNOWN result gives you meaningless numbers rather than an error.
    grid = None
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        grid = read_grid_from_solver(solver, cells)

    return solver, status, grid

In [50]:
def report(solver, status, label=""):
    """Print the statistics worth looking at after every solve."""
    gap = solver.best_objective_bound - solver.objective_value
    print(label)
    print(f"  status     : {solver.status_name(status)}")
    print(f"  objective  : {solver.objective_value}")
    print(f"  bound      : {solver.best_objective_bound}   (gap {gap:g})")
    print(f"  branches   : {solver.num_branches}")
    print(f"  conflicts  : {solver.num_conflicts}")
    print(f"  wall time  : {solver.wall_time:.4f}s")

In [51]:
model, cells, total = build_base()
model.maximize(total)

solver, status, grid = solve(model, cells)
report(solver, status, "BASE MODEL (rules only)")

show(grid, "\nBest grid found:")
print()
print_check(grid)

BASE MODEL (rules only)
  status     : OPTIMAL
  objective  : 205.0
  bound      : 205.0   (gap 0)
  branches   : 30
  conflicts  : 0
  wall time  : 0.0195s

Best grid found:
        ÷6   ÷7   ÷8   ÷9   ÷10  
  ÷1    9   8   9   9   9    = 98999
  ÷2    9   9   9   9   8    = 99998
  ÷3    7   9   8   9   9    = 79899
  ÷4    9   9   8   9   6    = 99896
  ÷5    8   9   8   9   0    = 89890

  col 1 (÷6 ) = 99798
  col 2 (÷7 ) = 89999
  col 3 (÷8 ) = 99888
  col 4 (÷9 ) = 99999
  col 5 (÷10) = 98960

  SUM OF DIGITS = 205

checker says: ✓ legal


### What the solver log actually says

Turn on `log_search_progress` and CP-SAT stops being a black box. Run the next cell and look for
these sections:

- **Initial model / presolve** — CP-SAT rewrites your model before solving: propagating fixed
  values, merging duplicate constraints, tightening bounds. It's normal for presolve to delete a
  large fraction of what you wrote. If presolve removes *everything*, your problem was trivial;
  if it removes nothing, you may have written it in a form the solver can't see through.
- **The `#1`, `#2`, `#Bound` lines** — each `#n` is a new incumbent solution, each `#Bound` a
  tightened proof ceiling. This is the gap closing in real time, and it's the thing to watch on a
  long run: if the bound isn't moving, more time probably won't help — you need a better model.
- **The `Task timing` table at the end** — which of the parallel strategies actually paid off
  (`default_lp`, `no_lp`, `max_lp`, `quick_restart`, feasibility jump heuristics, …). On hard
  problems this tells you what to lean on.

In [52]:
model, cells, total = build_base()
model.maximize(total)
solver, status, grid = solve(model, cells, log=True)


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 60 log_search_progress: true num_workers: 8

Initial optimization model '': (model_fingerprint: 0x4436583f2da693e1)
#Variables: 35 (#ints: 25 in objective) (26 primary variables)
  - 25 in [0,9]
  - 1 in [0,9999]
  - 1 in [0,11111]
  - 1 in [0,12499]
  - 1 in [0,14285]
  - 1 in [0,16666]
  - 1 in [0,19999]
  - 1 in [0,24999]
  - 1 in [0,33333]
  - 1 in [0,49999]
  - 1 in [0,99999]
#kLinearN: 10 (#terms: 60)

Starting presolve at 0.00s
  2.20e-05s  0.00e+00d  [DetectDominanceRelations] 
  1.46e-03s  0.00e+00d  [PresolveToFixPoint] #num_loops=3 #num_dual_strengthening=1 
  2.00e-06s  0.00e+00d  [ExtractEncodingFromLinear] 
  8.00e-06s  0.00e+00d  [DetectDuplicateColumns] 
  1.20e-05s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 81 nodes and 73 arcs.
[Symmetry] Symmetry computation done. time: 2.4e-05 dtime: 1.212e-05
  1.20e-05s  0.00e+00d  [DetectDuplicateConstraintsWithDifferentEnforcemen

## 6. Adding *your* knowledge: implied constraints

This is the workflow you actually want. You stare at a puzzle, notice something — *"the
bottom-right cell has to be 0"* — and you want to hand that to the solver.

There are two very different kinds of thing you can add, and confusing them is how people get
wrong answers:

**Implied (redundant) constraints** are facts *logically entailed* by the rules you already
wrote. They remove **zero** solutions. They are always safe. Their entire purpose is to give the
propagator something to chew on early, so it prunes at depth 2 instead of depth 15. Adding these
is the highest-leverage thing you can do to a slow model.

**Assumptions** are guesses that are *not* entailed — "I bet the top-left is 9." These genuinely
cut the search space, and if you're wrong you'll get a worse answer or `INFEASIBLE` with no
warning. Use them to explore, never in the model you trust. Section 6b shows how.

### Deriving the implied constraints

Let $d_{r,c}$ be the digit at row $r$, column $c$, zero-indexed.

**Column 5 is divisible by 10** $\Rightarrow$ its last digit is 0 $\Rightarrow$ $d_{4,4} = 0$.
That's your bottom-right-cell intuition, and it's provable. Note the solver *does* already know
this — it's a one-step consequence — but writing it costs nothing and it's the template for the
rest.

**Row 5 is divisible by 5** $\Rightarrow$ $d_{4,4} \in \{0, 5\}$. Combined with the above,
consistent, and no new information. Worth checking anyway: **when two deductions about the same
cell disagree, you've misread the puzzle.** Cheap insurance.

**Row 2 is divisible by 2** $\Rightarrow$ $d_{1,4}$ is even.

**Column 1 is divisible by 6** $= 2 \times 3$ $\Rightarrow$ $d_{4,0}$ is even, *and* the digit
sum of column 1 is divisible by 3.

**Row 3 is divisible by 3** $\Rightarrow$ its digit sum is divisible by 3. This one is genuinely
useful: it's a constraint on a *sum of five variables*, which propagates in a completely
different direction from the place-value equation. The solver has to work to derive it; you get
it free from a rule you learned in primary school.

**Column 4 is divisible by 9** $\Rightarrow$ its digit sum is divisible by 9. Same idea, and even
stronger — a five-digit sum is at most 45, so this pins the column's digit sum to
$\{0, 9, 18, 27, 36, 45\}$. Since we're *maximizing*, that's a hard, high-value cut.

**Row 4 is divisible by 4** $\Rightarrow$ its last two digits form a multiple of 4.

**Column 3 is divisible by 8** $\Rightarrow$ its last three digits form a multiple of 8.

Notice the pattern: the divisibility rules you know from mental arithmetic are exactly the
implied constraints worth writing. Sum-of-digits rules (3, 9) and last-$k$-digits rules
(2, 4, 5, 8, 10) cover almost everything.

In [53]:
def add_implied(model, cells):
    """Facts entailed by the rules. These remove zero solutions — they only speed up search."""

    # Column 5 is divisible by 10, so its last digit is 0.
    model.add(cells[4][4] == 0)

    # Row 5 is divisible by 5, so its last digit is 0 or 5. Consistent with the line
    # above; included as a cross-check that we read the clues correctly.
    model.add_allowed_assignments([cells[4][4]], [(0,), (5,)])

    # Row 2 is divisible by 2, so its last digit is even.
    force_divisible(model, cells[1][4], 2, 9, "row2_last_digit")

    # Column 1 is divisible by 6 = 2 x 3, which gives us two separate facts.
    force_divisible(model, cells[4][0], 2, 9, "col1_last_digit")
    force_divisible(
        model, column_digit_sum_expression(cells, 0), 3, 45, "col1_digit_sum"
    )

    # Row 3 is divisible by 3, so its digit sum is divisible by 3.
    force_divisible(model, row_digit_sum_expression(cells, 2), 3, 45, "row3_digit_sum")

    # Column 4 is divisible by 9, so its digit sum is divisible by 9. A five-digit sum
    # is at most 45, so this pins the sum to one of {0, 9, 18, 27, 36, 45}.
    force_divisible(
        model, column_digit_sum_expression(cells, 3), 9, 45, "col4_digit_sum"
    )

    # Row 4 is divisible by 4, so its last TWO digits form a multiple of 4.
    row4_tail = cells[3][3] * 10 + cells[3][4]
    force_divisible(model, row4_tail, 4, 99, "row4_tail")

    # Column 3 is divisible by 8, so its last THREE digits form a multiple of 8.
    col3_tail = cells[2][2] * 100 + cells[3][2] * 10 + cells[4][2]
    force_divisible(model, col3_tail, 8, 999, "col3_tail")


def build_smart():
    """The puzzle's rules plus our own deductions."""
    model, cells, total = build_base()
    add_implied(model, cells)
    return model, cells, total


print("add_implied() defined — 9 human deductions, all provably redundant")

add_implied() defined — 9 human deductions, all provably redundant


In [54]:
# Solve both models and compare. The objective MUST come out the same.
model_labels = ["base (rules only)", "+ implied constraints"]
model_builders = [build_base, build_smart]

solvers_by_label = {}
for i in range(len(model_labels)):
    label = model_labels[i]
    build = model_builders[i]

    model, cells, total = build()
    model.maximize(total)
    solver, status, grid = solve(model, cells)
    solvers_by_label[label] = (solver, status, grid)

header = f"{'model':<24} {'status':<10} {'obj':>6} {'bound':>7}"
header = header + f" {'branches':>10} {'conflicts':>10} {'time':>9}"
print(header)
print("-" * 80)

objectives = []
for label in model_labels:
    solver, status, grid = solvers_by_label[label]
    objectives.append(solver.objective_value)
    line = (
        f"{label:<24} {solver.status_name(status):<10} {solver.objective_value:>6.0f}"
    )
    line = line + f" {solver.best_objective_bound:>7.0f} {solver.num_branches:>10}"
    line = line + f" {solver.num_conflicts:>10} {solver.wall_time:>8.4f}s"
    print(line)

# If these disagree, one of the "implied" constraints was not actually implied — it cut
# off real solutions. That is a bug, and this assert is what catches it.
assert (
    objectives[0] == objectives[1]
), f"MISMATCH — an implied constraint cut solutions: {objectives}"

print("\n✓ Both models agree on the optimum. That equality is the proof that every")
print("  constraint we added really was implied — a redundant constraint that changes")
print("  the answer was never redundant, it was a bug.")

model                    status        obj   bound   branches  conflicts      time
--------------------------------------------------------------------------------
base (rules only)        OPTIMAL       205     205         84          0   0.0199s
+ implied constraints    OPTIMAL       205     205          0          0   0.0111s

✓ Both models agree on the optimum. That equality is the proof that every
  constraint we added really was implied — a redundant constraint that changes
  the answer was never redundant, it was a bug.


### Read that table carefully

The number to look at is **branches**. If it drops to zero (or near it) in the second row, that
means the solver never had to *guess* — propagation alone drove every variable to a single value.
Your deductions did the whole job; the search tree collapsed to a point.

On a puzzle this small the wall-clock difference is noise — both models finish in milliseconds
either way, and honestly you didn't need the implied constraints at all. **That is the point of
practising it here.** On a harder puzzle (Jane Street's later grid puzzles absolutely qualify)
the same technique is the difference between three seconds and never finishing, and you want the
habit already built when you get there. You'll see the effect properly in section 7, where the
enumeration workload is large enough to measure.

And note the assertion at the end of that cell. **Every time you add an "implied" constraint,
re-solve and check the objective didn't move.** It's a two-line test that catches the single most
dangerous class of modeling bug, where a plausible-looking deduction is subtly wrong and quietly
throws away the real answer.

### 6b. Assumptions — exploring "what if?" safely

Sometimes you want to test a hunch that *isn't* entailed: *what's the best grid where the top-left
digit is 9?* The rule is simple — **never add an assumption to a model you intend to trust.**
Build a fresh model, add the guess there, and compare against your known optimum.

This is also how you do sensitivity analysis: fix a cell to each of its 10 values in turn and see
what the best achievable total is under each. That tells you which cells the optimum is actually
sensitive to.

In [55]:
best_known = int(solvers_by_label["+ implied constraints"][0].objective_value)

print(f"unconstrained optimum = {best_known}\n")
print("best total achievable when the TOP-LEFT cell is forced to each digit:")

for guess in range(10):
    model, cells, total = build_smart()
    model.add(cells[0][0] == guess)  # an ASSUMPTION, in a throwaway model
    model.maximize(total)
    solver, status, grid = solve(model, cells, time_limit=10)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        best_here = int(solver.objective_value)
        cost = best_known - best_here
        if cost == 0:
            note = "  <-- optimal"
        else:
            note = f"  (costs {cost})"
        print(f"  cells[0][0] = {guess}  ->  best total {best_here}{note}")
    else:
        print(f"  cells[0][0] = {guess}  ->  INFEASIBLE (no legal grid at all)")

unconstrained optimum = 205

best total achievable when the TOP-LEFT cell is forced to each digit:
  cells[0][0] = 0  ->  best total 196  (costs 9)
  cells[0][0] = 1  ->  best total 198  (costs 7)
  cells[0][0] = 2  ->  best total 198  (costs 7)
  cells[0][0] = 3  ->  best total 199  (costs 6)
  cells[0][0] = 4  ->  best total 201  (costs 4)
  cells[0][0] = 5  ->  best total 201  (costs 4)
  cells[0][0] = 6  ->  best total 202  (costs 3)
  cells[0][0] = 7  ->  best total 204  (costs 1)
  cells[0][0] = 8  ->  best total 204  (costs 1)
  cells[0][0] = 9  ->  best total 205  <-- optimal


## 7. Enumerating the pruned space

You asked for this specifically: not one answer, but *the whole set of survivors*. This is where
solvers stop being a calculator and start being a microscope.

### The gotcha that trips everyone up

**CP-SAT will not enumerate solutions while an objective is set.** With `maximize()` in the model,
`enumerate_all_solutions` is ignored — the two features are mutually exclusive, and it fails
quietly rather than erroring. The fix is a two-step pattern you'll use constantly:

1. Solve once *with* the objective to learn the optimum $B$.
2. Build the model again *without* the objective, add the hard constraint `total == B`,
   and enumerate.

You've turned an optimization problem into a satisfaction problem. Now every solution the solver
finds is an optimal one.

### The other two settings

- `enumerate_all_solutions = True`
- `num_workers = 1` — enumeration needs the single-threaded search. The parallel portfolio would
  otherwise report duplicates and its own workers would prune each other's solutions.

Solutions arrive through a **callback**, invoked once per solution found, rather than as a return
value. Subclass `CpSolverSolutionCallback` and read variables with `self.value(variable)`.

In [56]:
class GridCollector(cp_model.CpSolverSolutionCallback):
    """Called once per solution found. Collects grids, with an optional cap."""

    def __init__(self, cells, limit=None):
        super().__init__()
        self.cells = cells
        self.limit = limit
        self.grids = []

    def on_solution_callback(self):
        grid = []
        for r in range(N):
            row = []
            for c in range(N):
                row.append(self.value(self.cells[r][c]))
            grid.append(row)
        self.grids.append(grid)

        if self.limit is not None and len(self.grids) >= self.limit:
            self.stop_search()

In [57]:
def enumerate_at(target_sum, limit=None, time_limit=60, use_implied=True):
    """Every legal grid whose digits sum to exactly target_sum.

    Returns (grids, solver, complete) where `complete` says whether the search was
    exhaustive — if it hit the time limit or the cap, the list is only a sample.
    """
    if use_implied:
        model, cells, total = build_smart()
    else:
        model, cells, total = build_base()

    # The objective becomes a constraint. Note we never call model.maximize() here:
    # an objective would silently disable enumeration.
    model.add(total == target_sum)

    collector = GridCollector(cells, limit)

    solver = cp_model.CpSolver()
    solver.parameters.enumerate_all_solutions = True  # needs no objective in the model
    solver.parameters.num_workers = 1  # required for correct enumeration
    solver.parameters.max_time_in_seconds = time_limit
    status = solver.solve(model, collector)

    complete = status == cp_model.OPTIMAL and limit is None
    return collector.grids, solver, complete

In [ ]:
grids, solver, complete = enumerate_at(best_known)

print(f"grids achieving the maximum sum of {best_known}: {len(grids)}")
print(f"search complete (exhaustive): {complete}   [{solver.wall_time:.3f}s]\n")

for grid in grids:
    show(grid)
    print()
    print_check(grid)
    print("=" * 50)

### The shape of the feasible region

One optimum is a fact. The *distribution* of solutions near the optimum is understanding — it
tells you whether the answer was a knife-edge or one of many, which is exactly what you want to
know before you trust a submission.

Sweep the target sum downward and count. Watch the counts explode: this is combinatorial
explosion happening in front of you, and it's why the objective-as-constraint trick is so
powerful. Constraining `total == 205` leaves a search space you can enumerate instantly;
`total == 195` would take a very long time.

In [58]:
DEPTH = 6  # raise this if you're patient — the cost roughly triples per step

print(f"{'sum':>5} {'grids':>8} {'time':>9}   distribution (log scale)")
print("-" * 62)

counts_by_sum = {}
for delta in range(DEPTH):
    target = best_known - delta
    grids_found, solver, complete = enumerate_at(target, time_limit=120)
    counts_by_sum[target] = len(grids_found)

    bar_length = int(6 * math.log10(len(grids_found) + 1)) + 1
    bar = "█" * bar_length
    if complete:
        note = ""
    else:
        note = "  (TIME LIMIT — count is a lower bound)"
    print(f"{target:>5} {len(grids_found):>8} {solver.wall_time:>8.2f}s   {bar}{note}")

grand_total = 0
for target in counts_by_sum:
    grand_total = grand_total + counts_by_sum[target]
print(f"\ntotal legal grids within {DEPTH - 1} of the optimum: {grand_total}")

  sum    grids      time   distribution (log scale)
--------------------------------------------------------------
  205        1     0.01s   ██
  204       22     0.03s   █████████
  203      120     0.03s   █████████████
  202      483     0.13s   █████████████████
  201     1659     0.47s   ████████████████████
  200     5038     2.53s   ███████████████████████

total legal grids within 5 of the optimum: 7323


### Measuring what the implied constraints actually bought us

Now that there's real work to do, we can put a number on section 6. Enumerate the *same* solution
set twice — once from the base model, once from the model with our deductions — and compare the
search statistics.

The two things to look for: the **counts must be identical** (that's the correctness check —
implied constraints cannot change which grids exist), while the **branches and conflicts should
drop** (that's the payoff).

In [59]:
target = best_known - 5
print(f"enumerating every legal grid with digit sum {target}, both ways:\n")
print(f"{'model':<24} {'grids':>8} {'branches':>10} {'conflicts':>10} {'time':>9}")
print("-" * 66)

counts = []
for label, use_implied in [
    ("base (rules only)", False),
    ("+ implied constraints", True),
]:
    grids_found, solver, complete = enumerate_at(
        target, time_limit=300, use_implied=use_implied
    )
    counts.append(len(grids_found))
    line = f"{label:<24} {len(grids_found):>8} {solver.num_branches:>10}"
    line = line + f" {solver.num_conflicts:>10} {solver.wall_time:>8.2f}s"
    print(line)

assert counts[0] == counts[1], "counts differ — an implied constraint was not implied!"
print(
    f"\n✓ identical solution counts ({counts[0]}) — the deductions cut search, not solutions"
)

enumerating every legal grid with digit sum 200, both ways:

model                       grids   branches  conflicts      time
------------------------------------------------------------------
base (rules only)            5038     185630      28338     2.85s
+ implied constraints        5038     173233       9978     2.56s

✓ identical solution counts (5038) — the deductions cut search, not solutions


## 8. Verify independently, then format the answer

The checker from section 3 has been sitting there the whole time. Use it on **everything** the
solver produced — not because CP-SAT is unreliable (it isn't), but because *your encoding of the
rules* might be. The checker and the model were written from the puzzle text separately; if they
agree on thousands of grids, the odds that both contain the same mistake are small.

In [60]:
# Gather every grid within 2 of the optimum.
all_grids = []
for delta in range(3):
    grids_found, solver, complete = enumerate_at(best_known - delta)
    for grid in grids_found:
        all_grids.append(grid)

# Run the independent checker over all of them.
violations = []
for grid in all_grids:
    problems = check(grid)
    if len(problems) > 0:
        violations.append((grid, problems))

print(f"independently verified {len(all_grids)} grids — {len(violations)} violations")
assert len(violations) == 0, violations[:3]

independently verified 143 grids — 0 violations


In [61]:
best_grid = grids[0]

digit_text = ""
for r in range(N):
    for c in range(N):
        digit_text = digit_text + str(best_grid[r][c])

answer = f"({digit_sum(best_grid)},{digit_text})"
print("SUBMISSION: ", answer)

SUBMISSION:  (205,9899999998798999989689890)


## 9. Best-practice cheat sheet

Everything this notebook demonstrated, condensed. This is the part to re-read before your next
puzzle.

**Modeling**

1. **Write the checker first**, and validate it on a known-good instance. A model is write-only
   code; the checker is your only ground truth.
2. **Keep domains tight.** `new_int_var(0, 9, ...)` not `new_int_var(0, 1000, ...)`. Bounds are
   free propagation.
3. **Prefer linear encodings.** `x == k * q` beats `x % k == 0`. Prefer a specialised global
   constraint (`add_all_different`, `add_circuit`, `add_element`) over hand-rolling one out of
   booleans — the specialised propagators are dramatically stronger. Reaching for `add_circuit`
   for path/loop puzzles is a big one for Jane Street grids specifically.
4. **Give every derived variable its own name.** Sharing one quotient across two divisibility
   rules silently forces both values to zero.
5. **Model the rules, not the search.** If you find yourself writing loops that try values,
   you're fighting the tool.

**Combining rules with boolean logic**

6. To express "rule A **or** rule B", *reify* each into a boolean with `.only_enforce_if(flag)`,
   then combine the flags: `model.add_bool_or([flag_a, flag_b])`.
7. Booleans behave as 0/1 integers in linear constraints, so counting is easy:
   `model.add(flag_a + flag_b == 1)` for exactly one, `>= 2` for at least two, and so on.
8. `only_enforce_if` gives you `flag -> constraint` but **not** the reverse. If you need to
   count flags or negate one, reify fully (both directions) via a remainder variable.
9. Only reify **independent** facts. A flag whose value is computable from the others adds
   variables and constraints for nothing.

**Speed**

10. **Add implied constraints** — your own deductions, written as extra constraints. Free speed,
    zero risk *if* they're genuinely entailed.
11. **Assert the objective doesn't move** after adding them. Two lines, catches the worst bug class.
12. **Break symmetry** when the problem has it. If a puzzle is invariant under, say, reflection,
    the solver will otherwise explore every mirror image of every dead end. (This puzzle has no
    symmetry — the divisors differ per row and column, so every cell is distinguishable.)
13. **Give hints** with `model.add_hint(variable, value)` when you have a decent guess or a
    previous solution. It seeds the incumbent so the bound starts closing immediately.
14. **`num_workers = 8`** for optimization — it's a portfolio of *different strategies* sharing
    learned clauses, not just parallel grunt. But use `1` when benchmarking, since the parallel
    statistics are non-deterministic.
15. **Always set `max_time_in_seconds`.** Non-negotiable in a notebook.

**Enumeration**

16. Objective and `enumerate_all_solutions` are **mutually exclusive**. Solve for the optimum $B$,
    then re-build with `total == B` as a constraint.
17. Enumeration requires **`num_workers = 1`**.
18. Cap the collector (`limit=`) before enumerating something you haven't sized. Counts explode fast.

**Debugging**

19. `INFEASIBLE` on a puzzle you know has an answer means **you made an encoding error** — that's
    the default assumption, not a solver bug. Comment out constraints one at a time until it
    becomes feasible; the last one you removed is the broken one.
20. **A missing `model.add()` fails silently.** Writing `row_value == 5 * quotient` on its own is
    legal Python that builds an object and discards it. If a rule seems to be ignored, look here.
21. `solver.parameters.log_search_progress = True` when anything is slow. If the **bound isn't
    moving**, more time won't save you — you need a better model.
22. `print(model.model_stats())` shows what your model actually contains after you built it.

## 10. Which solver next time?

CP-SAT is the right default for most Jane Street puzzles, but knowing when it isn't matters.

| Your problem | Reach for | Why |
|---|---|---|
| Integer/finite variables, combinatorial rules, maybe an objective | **OR-Tools CP-SAT** | The default. Grid puzzles, packing, scheduling, routing, path-finding. Covers ~90% of these. |
| Continuous variables, linear objective | **HiGHS** (or OR-Tools' MIP wrapper) | LP/MIP territory. CP-SAT is integer-only. |
| Pure logic, quantifiers, bitvectors, program semantics | **Z3** (SMT) | When "solve" means "prove", not "optimize". |
| Search space under ~$10^8$ and clarity beats speed | **Plain Python DFS** | Don't reach for machinery you don't need. A readable brute force you trust beats a clever model you don't. |
| You want to write the model once and try many backends | **CPMpy** | A numpy-flavoured modeling layer over CP-SAT, Z3, Gurobi, MiniZinc and others. Nice once CP-SAT is second nature — learn the engine first. |

**A reusable template for the next puzzle**, in the order to write it:

```
1. read the puzzle, list the rules in English
2. write check(solution) in plain Python
3. find or construct one known-good instance; assert check() accepts it
4. choose decision variables (this is the real modeling skill — pick the
   representation where the rules are easiest to state)
5. encode each rule as a constraint; re-read your English list and tick them off
6. solve; confirm OPTIMAL and gap == 0
7. add your own deductions as implied constraints; assert the objective is unchanged
8. drop the objective, constrain total == optimum, enumerate
9. run check() on everything before you submit
```

Step 4 is where the craft lives. Same puzzle, different variable choice, thousand-fold difference
in solve time. Here, digits-as-variables was obvious — but you could equally have used five
"column value" variables ranging over multiples of their divisor, linked to the digits. When a
model is slow, changing the variables is usually a bigger lever than adding constraints.

**Bookmarks worth having:**

- [CP-SAT primer](https://developers.google.com/optimization/cp/cp_solver) — the official guide
- [The CP-SAT Primer by Krupke](https://d-krupke.github.io/cpsat-primer/) — much deeper, the best
  free resource on getting real performance out of this solver
- [Python reference for `cp_model`](https://developers.google.com/optimization/reference/python/sat/python/cp_model)